In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=False,
    c1=1e-4,
    tau=0.5,
    line_search_method="backtrack",
    line_search_cond="armijo",
    # batch_size=100
    batch_size=None
)

all_loss = []
patience = 0
max_patience = 10
for epoch in range(100):
    print("epoch: ", epoch, end="")
    all_loss.append(0)
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch] += loss

    all_loss[epoch] /= len(data_loader)

    if epoch > 0 and all_loss[epoch - 1] <= all_loss[epoch]:
        patience -= 1
    else:
        patience = max_patience

    print(", loss: {}".format(all_loss[epoch].detach().cpu().numpy().item()))

    if patience <= 0:
        break

epoch:  0, loss: 0.39818572998046875
epoch:  1, loss: 0.09859157353639603
epoch:  2, loss: 0.06208353117108345
epoch:  3, loss: 0.015426053665578365
epoch:  4, loss: 0.009946099482476711
epoch:  5, loss: 0.009701551869511604
epoch:  6, loss: 0.008133544586598873
epoch:  7, loss: 0.008087685331702232
epoch:  8, loss: 0.007829071022570133
epoch:  9, loss: 0.007756099104881287
epoch:  10, loss: 0.0077115916647017
epoch:  11, loss: 0.0077001177705824375
epoch:  12, loss: 0.007696990855038166
epoch:  13, loss: 0.007148304022848606
epoch:  14, loss: 0.006650602910667658
epoch:  15, loss: 0.006390864960849285
epoch:  16, loss: 0.005658013746142387
epoch:  17, loss: 0.005145565140992403
epoch:  18, loss: 0.005120945628732443
epoch:  19, loss: 0.004419305827468634
epoch:  20, loss: 0.004407192580401897
epoch:  21, loss: 0.00431947922334075
epoch:  22, loss: 0.003953584004193544
epoch:  23, loss: 0.0038950657472014427
epoch:  24, loss: 0.003883199766278267
epoch:  25, loss: 0.003711333265528083


In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9331457876521544
Test metrics:  R2 = 0.9287981125963547


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100
)

all_loss = []
patience = 0
max_patience = 10
for epoch in range(100):
    print("epoch: ", epoch, end="")
    all_loss.append(0)
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch] += loss

    all_loss[epoch] /= len(data_loader)

    if epoch > 0 and all_loss[epoch - 1] <= all_loss[epoch]:
        patience -= 1
    else:
        patience = max_patience

    print(", loss: {}".format(all_loss[epoch].cpu().detach().numpy().item()))

    if patience <= 0:
        break

epoch:  0, loss: 0.47211581468582153
epoch:  1, loss: 0.15253639221191406
epoch:  2, loss: 0.032296322286129
epoch:  3, loss: 0.009960396215319633
epoch:  4, loss: 0.00829842034727335
epoch:  5, loss: 0.0078006768599152565
epoch:  6, loss: 0.007590240333229303
epoch:  7, loss: 0.007409481797367334
epoch:  8, loss: 0.006983920466154814
epoch:  9, loss: 0.00674658827483654
epoch:  10, loss: 0.006578656379133463
epoch:  11, loss: 0.006419877987354994
epoch:  12, loss: 0.006273524835705757
epoch:  13, loss: 0.006109657697379589
epoch:  14, loss: 0.00595358619466424
epoch:  15, loss: 0.00584239000454545
epoch:  16, loss: 0.005705688614398241
epoch:  17, loss: 0.005532356444746256
epoch:  18, loss: 0.0053303479216992855
epoch:  19, loss: 0.005076847039163113
epoch:  20, loss: 0.00472002848982811
epoch:  21, loss: 0.004369194153696299
epoch:  22, loss: 0.004057272803038359
epoch:  23, loss: 0.003796081757172942
epoch:  24, loss: 0.003572314279153943
epoch:  25, loss: 0.003380722366273403
epoc

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.941890793468452
Test metrics:  R2 = 0.938401760564126
